# Multi-sample Fresnel ellipsometry workflow

This notebook is a **template**. Replace the example paths, stack, material names, and thickness ranges with your experiment. The unknown material is the finite layer for which no optical-constant file is supplied.

Use `FIT_MODE = "spline"` for adaptive KK-consistent B-spline reconstruction or `FIT_MODE = "physical"` for sequential physical-model selection. Multiple samples share one dielectric function while retaining independent thicknesses.

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from fresnel_ellipsometry import (
    MultiFitDataset,
    read_material_data, read_ellipsometry_data,
    experimental_wavelength_grid, smooth_and_interpolate_exp,
    interpolate_dielectric_function, measurement_bandwidth,
    build_stack_model, build_measurement_data,
    configure_multifit_thickness_parameters, multifit_backend_info,
    build_initial_spline, build_initial_physical_model,
    build_fit_parameters, build_multi_fit_parameters,
    build_physical_parameters, build_multi_physical_parameters,
    build_thickness_parameters, build_multi_thickness_parameters,
    automatic_thickness_prefit, varying_thickness_names, print_thickness_prefit_summary,
    clear_thickness_anchors, seed_fitted_thicknesses, set_thickness_anchors,
    set_dielectric_guard,
    adaptive_fit_spline, adaptive_fit_spline_multi,
    adaptive_fit_physical, adaptive_fit_physical_multi,
    fit_thicknesses, fit_thicknesses_multi,
    multi_dataset_fit_table, thickness_fit_table, print_physical_fit_report,
    refractive_index_with_uncertainty,
    plot_loaded_datasets, plot_fit, plot_residual_evolution, plot_refractive_index,
    save_refractive_index_csv,
)

### 1. Build experiment configuration

Here we build the stack, load experimental data (with thickness constraints) and load known optical constants. Known uniaxial materials are indexed `<material>_e` and `<material>_o`.


In [2]:
# Define stack and data constraints here
stack = "Air/PPA/Flake/SiO2/Si"
symmetry = "uniaxial"  # "isotropic" or "uniaxial"

measurement_dir = Path("data/measurements")
index_data_dir = Path("data/optical_constants")

# Scalar = fixed thickness. Two values = fitted [lower, upper] interval.
files = {
"dataexample.ods": {
    "PPA": np.array([250, 400]) * 1e-9,
    "Flake": np.array([60, 90]) * 1e-9,
    "SiO2": 90e-9,
},
# "file2.ods": {
#     "Polymer": np.array([100, 150]) * 1e-9,
#     "Flake": np.array([10, 20]) * 1e-9,
#     "SiO2": 90e-9,
# },
}

spectral_weighting = "points"  # "points", "wavelength", or "energy"
RESULTS_DIR = Path("results")
HC_EV_NM = 1239.8419843320026

### 2. Data shaping and fit type
Optional cropping, smoothing and interpolation of the experimental data is defined here. Additionally, FIT_MODE desides between B-spline function reconstruction, or dispersion model based reconstruction.

In [ ]:
FIT_MODE = "spline"           # "spline" or "physical"
AOIS = None                    # None -> use all measured AOIs, otherwise specify a list of AOIs to use (e.g. AOIS = [50, 60, 70])
wl_range_nm = (0, 1200)         # Define wavelength range to load
smoothing_nm = False  # Gaussian smoothing of data. (e.g. smoothing_nm = 5). False if no smoothing
interpolation_nm = False # Spline interpolation of data. Will interpolate to set wavelength density. (e.g. interpolation_nm = 5) False for no interpolation

material_names = stack.split("/")
layers = []
for i, name in enumerate(material_names):
    try:
        layers.append(read_material_data(index_data_dir / name))
    except FileNotFoundError:
        if i in (0, len(material_names) - 1):
            raise
        layers.append(None)


def resolve_measurement(filename):
    path = measurement_dir / filename
    if not path.exists():
        raise FileNotFoundError(path)
    return path

required_columns = {"lambda_(nm)", "AOI_(°)", "delta_(°)", "delta_sigma_(°)","psi_(°)", "psi_sigma_(°)",}

loaded = {}
preprocessed = {}
for filename, thicknesses in files.items():
    data = read_ellipsometry_data(resolve_measurement(filename))
    missing = sorted(required_columns - set(data.columns))
    if missing:
        raise ValueError(f"{filename}: missing columns {missing}")
    aois = tuple(AOIS or np.sort(data["AOI_(°)"].unique()))
    data = data[data["AOI_(°)"].isin(aois) & data["lambda_(nm)"].between(*wl_range_nm)].copy()
    wl_nm, interpolation_applied, _ = experimental_wavelength_grid(data, aois, interpolation=interpolation_nm)
    meas = {aoi: smooth_and_interpolate_exp(data, aoi, wl_nm,smooth=smoothing_nm,interpolate=interpolation_applied,) for aoi in aois}
    print(f"{filename}: loaded {len(aois)} AOI(s), {wl_nm.size} wavelengths "f"from {wl_nm.min():.1f} to {wl_nm.max():.1f} nm | "f"smoothing={smoothing_nm}, interpolation={interpolation_nm}") 
    preprocessed[filename] = {"meas": meas, "AOIs": aois, "wl_nm": wl_nm}

    dielectric_functions = {}
    for i, (name, layer_data) in enumerate(zip(material_names, layers)):
        if layer_data is not None:
            dielectric_functions[f"layer_{i}"] = interpolate_dielectric_function(
                layer_data, wl_nm, material_name=name
            )

    stack_model = build_stack_model(
        stack, dielectric_functions, thicknesses, wl_nm, symmetry=symmetry
    )
    measurement = build_measurement_data(
        meas, aois, wl_nm,
        spectral_weighting=spectral_weighting,
        hc_ev_nm=HC_EV_NM,
    )
    bandwidth_nm = measurement_bandwidth(data, wl_nm)
    loaded[filename] = MultiFitDataset(
        filename, stack_model, measurement, bandwidth_nm
    )

plot_loaded_datasets(preprocessed)
plt.show()

fit_datasets = list(loaded.values())
multi_sample = len(fit_datasets) > 1
if multi_sample:
    configure_multifit_thickness_parameters(fit_datasets)
    print("Multi-sample backend enabled", multifit_backend_info(fit_datasets))

### 3. Initialize the dielectric model

The initial model is intentionally low-dimensional. Thickness prefitting is optional but useful when the dielectric response and structural parameters are strongly correlated.

In [5]:
# These are dielectric function constraints. Generally keep them as unconstrained as possible. 
EPS_OFFSET_BOUNDS = (-np.inf, np.inf)
EPS2_COEFF_BOUNDS = (0.0, np.inf)
EPS_OFFSET_O_INIT = 3.0
EPS_OFFSET_E_INIT = 3.0
EPS2_COEFF_INIT = 1e-3
INITIAL_FREE_NODES = 4

PHYSICAL_EPS_INF_BOUNDS = (-50.0, 100.0) # In principle -inf to inf, this helps convergence however.
PHYSICAL_EPS_INF_O_INIT = 3.0
PHYSICAL_EPS_INF_E_INIT = 3.0

DIELECTRIC_GUARD = True
MAX_COMPLEX_INDEX = 20.0 # Soft limit on maximum complex index. Final value can exceed this.

unknown_flags = [d.stack_model.has_unknown for d in fit_datasets]
if any(unknown_flags) and not all(unknown_flags):
    raise ValueError("Do not mix known-only and reconstruction datasets in one fit.")
all_materials_known = not any(unknown_flags)
stack_model = fit_datasets[0].stack_model

if DIELECTRIC_GUARD:
    for dataset in fit_datasets:
        if dataset.stack_model.has_unknown:
            set_dielectric_guard(dataset.stack_model, max_index=MAX_COMPLEX_INDEX)

optical_model = None
spline = None
physical_model = None

if all_materials_known:
    params = (
        build_multi_thickness_parameters(fit_datasets)
        if multi_sample else build_thickness_parameters(stack_model)
    )
else:
    model_wl_nm = np.unique(np.concatenate([d.measurement.wl_nm for d in fit_datasets]))
    bandwidth_curves = []
    for dataset in fit_datasets:
        curve = np.full(model_wl_nm.shape, np.nan)
        wl = dataset.measurement.wl_nm
        inside = (model_wl_nm >= wl.min()) & (model_wl_nm <= wl.max())
        curve[inside] = np.interp(model_wl_nm[inside], wl, dataset.bandwidth_nm)
        bandwidth_curves.append(curve)
    model_bandwidth_nm = np.nanmedian(np.stack(bandwidth_curves), axis=0)
    finite = model_bandwidth_nm[np.isfinite(model_bandwidth_nm)]
    model_bandwidth_nm = np.where(np.isfinite(model_bandwidth_nm), model_bandwidth_nm, np.median(finite))
    energy_eV = HC_EV_NM / model_wl_nm

    if FIT_MODE == "spline":
        spline = build_initial_spline(
            model_wl_nm, model_bandwidth_nm,
            degree=3,
            initial_free_nodes=INITIAL_FREE_NODES,
            kk_e_min=max(0.05, 0.30 * np.min(energy_eV)),
            kk_e_max=2.50 * np.max(energy_eV),
            kk_points=1201,
        )
        optical_model = spline
        params = (build_multi_fit_parameters(fit_datasets, spline, EPS_OFFSET_BOUNDS, EPS2_COEFF_BOUNDS,EPS_OFFSET_O_INIT, EPS_OFFSET_E_INIT, EPS2_COEFF_INIT,) if multi_sample else
            build_fit_parameters(
                stack_model, spline, EPS_OFFSET_BOUNDS, EPS2_COEFF_BOUNDS,
                EPS_OFFSET_O_INIT, EPS_OFFSET_E_INIT, EPS2_COEFF_INIT,
            )
        )
    else:
        physical_model = build_initial_physical_model(
            model_wl_nm,
            kk_e_min=max(0.01, 0.10 * np.min(energy_eV)),
            kk_e_max=3.00 * np.max(energy_eV),
            kk_points=1201,
            hc_ev_nm=HC_EV_NM,
        )
        optical_model = physical_model
        params = (
            build_multi_physical_parameters(
                fit_datasets, physical_model,
                eps_inf_bounds=PHYSICAL_EPS_INF_BOUNDS,
                eps_inf_parallel_init=PHYSICAL_EPS_INF_O_INIT,
                eps_inf_perp_init=PHYSICAL_EPS_INF_E_INIT,
            ) if multi_sample else
            build_physical_parameters(
                stack_model, physical_model,
                eps_inf_bounds=PHYSICAL_EPS_INF_BOUNDS,
                eps_inf_parallel_init=PHYSICAL_EPS_INF_O_INIT,
                eps_inf_perp_init=PHYSICAL_EPS_INF_E_INIT,
            )
        )

In [ ]:
RUN_THICKNESS_PREFIT = True
THICKNESS_SOFT_ANCHOR = False

if not all_materials_known and RUN_THICKNESS_PREFIT:
    for dataset in fit_datasets:
        clear_thickness_anchors(dataset.stack_model)
        free_names = varying_thickness_names(dataset.stack_model)
        if not free_names:
            continue
        prefit = automatic_thickness_prefit(
            dataset.stack_model, dataset.measurement, dataset.bandwidth_nm,
            eps_offset_bounds=EPS_OFFSET_BOUNDS,
            eps2_coeff_bounds=EPS2_COEFF_BOUNDS,
            eps_offset_parallel_init=EPS_OFFSET_O_INIT,
            eps_offset_perp_init=EPS_OFFSET_E_INIT,
            eps2_coeff_init=EPS2_COEFF_INIT,
            degree=3, window_nm=150.0, step_nm=50.0,
            node_options=(2, 3), target_rms_sigma=1.5,
            max_nfev=2000, min_points=20, kk_points=501,
        )
        print_thickness_prefit_summary(prefit, label=dataset.label)
        free_names = varying_thickness_names(dataset.stack_model)
        seed_fitted_thicknesses(dataset.stack_model, prefit["best"]["result"].params, free_names)
        if THICKNESS_SOFT_ANCHOR:
            set_thickness_anchors(dataset.stack_model, prefit["best"]["result"], free_names)

    # Rebuild parameters after updating thickness seeds without changing the dielectric-model bounds or initialization policy.
    if FIT_MODE == "spline":
        params = (
            build_multi_fit_parameters(
                fit_datasets, spline, EPS_OFFSET_BOUNDS, EPS2_COEFF_BOUNDS,
                EPS_OFFSET_O_INIT, EPS_OFFSET_E_INIT, EPS2_COEFF_INIT,
            ) if multi_sample else
            build_fit_parameters(
                stack_model, spline, EPS_OFFSET_BOUNDS, EPS2_COEFF_BOUNDS,
                EPS_OFFSET_O_INIT, EPS_OFFSET_E_INIT, EPS2_COEFF_INIT,
            )
        )
    else:
        params = (
            build_multi_physical_parameters(
                fit_datasets, physical_model,
                eps_inf_bounds=PHYSICAL_EPS_INF_BOUNDS,
                eps_inf_parallel_init=PHYSICAL_EPS_INF_O_INIT,
                eps_inf_perp_init=PHYSICAL_EPS_INF_E_INIT,
            ) if multi_sample else
            build_physical_parameters(
                stack_model, physical_model,
                eps_inf_bounds=PHYSICAL_EPS_INF_BOUNDS,
                eps_inf_parallel_init=PHYSICAL_EPS_INF_O_INIT,
                eps_inf_perp_init=PHYSICAL_EPS_INF_E_INIT,
            )
        )


### 4. Fit

pray.

In [ ]:
max_physical_terms = 4 # Physical fitting is slow, keep this number small
lineshape_candidates = ("cauchy", "sellmeier_uv", "sellmeier_ir", "drude", # Remove candidates if they are surely wrong
    "lorentz", "harmonic", "gaussian", "tauc_lorentz", "cody_lorentz",)

if all_materials_known:
    fit_snapshots = []
    optical_model = None
    fit_result = (
        fit_thicknesses_multi(params, fit_datasets, max_nfev=100000)
        if multi_sample else
        fit_thicknesses(params, stack_model, fit_datasets[0].measurement, max_nfev=100000)
    )
elif FIT_MODE == "spline":
    kwargs = dict(
        max_iterations=15, max_free_nodes=50, max_nfev=100000, # Might be worth changing iterations
        trial_max_nfev=600, max_single_candidates=3,
        residual_threshold=1.5, individual_residual_threshold=4.0,
        bic_improvement=1e4, min_spacing_factor=0.5, # Might have to increase/decrease bic_improvement.
        residual_smooth_factor=0.5, bootstrap=True,
        initial_multistart=True,
        multistart_n_starts=10,
        multistart_screen_max_nfev=250,
        multistart_n_finalists=2,
        verbose=False, show_progress=True,
    )
    if multi_sample:
        fit_result, spline, fit_history, fit_snapshots = adaptive_fit_spline_multi(params, spline, fit_datasets, **kwargs)
    else:
        d = fit_datasets[0]
        fit_result, spline, fit_history, fit_snapshots = adaptive_fit_spline(params, spline, d.stack_model, d.measurement, d.bandwidth_nm, **kwargs)
    optical_model = spline
else:
    candidates = lineshape_candidates
    kwargs = dict(
        candidate_types=candidates, max_terms=max_physical_terms, max_nfev=100000,
        trial_max_nfev=600, screen_max_nfev=180, n_family_finalists=4,
        bic_improvement=1e4, residual_threshold=1.5,
        individual_residual_threshold=4.0, residual_smooth_factor=0.5,
        n_centers=3, verbose=False, show_progress=True,
    )
    if multi_sample:
        fit_result, physical_model, fit_history, fit_snapshots = adaptive_fit_physical_multi(params, physical_model, fit_datasets, **kwargs)
    else:
        d = fit_datasets[0]
        fit_result, physical_model, fit_history, fit_snapshots = adaptive_fit_physical(params, physical_model, d.stack_model, d.measurement, d.bandwidth_nm, **kwargs)
    optical_model = physical_model

print(f"RMS = {fit_result.sigma_rms:.3f} sigma")
if multi_sample:
    display(multi_dataset_fit_table(fit_result, fit_datasets))
else:
    display(thickness_fit_table(fit_result, fit_datasets[0].stack_model))
if FIT_MODE == "physical" and not all_materials_known:
    print_physical_fit_report(fit_result, physical_model, stack_model)

In [ ]:
SAVE_CSV = False

plot_fit(fit_result, fit_datasets, optical_model)
plt.show()
if fit_snapshots:
    plot_residual_evolution(fit_snapshots, fit_datasets)
    plt.show()
if not all_materials_known:
    index_fit = refractive_index_with_uncertainty(fit_result, optical_model, stack_model)
    plot_refractive_index(model_wl_nm, index_fit)
    plt.show()
if not all_materials_known and SAVE_CSV:
    written = save_refractive_index_csv(model_wl_nm,index_fit,stack_model.unknown_name,output_dir=RESULTS_DIR,)
    for output_path in written:
        print(f"Saved fitted refractive index to {output_path}")
